# 01 - Multimodal Data Acquisition, Integration & Preprocessing
## AI-Based Cyber Risk Detection System

This notebook demonstrates the end-to-end data pipeline:
1. **Multimodal Data Acquisition & Integration** (CICIDS2017, UNSW-NB15, NVD/CVE, Synthetic Telemetry).
2. **Data Cleaning & Normalization** (missing value imputation, IQR outlier removal, standard scaling).
3. **Class Imbalance Mitigation** using SMOTE, BorderlineSMOTE, and ADASYN.
4. **Feature Selection** via Mutual Information, ANOVA F-test, and RFE.
5. **Dimensionality Reduction** comparing PCA, LDA, and TruncatedSVD.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.synthetic import SyntheticDataGenerator
from src.data.integration import DataIntegrator
from src.preprocessing.cleaner import DataCleaner
from src.preprocessing.transformer import DataTransformer
from src.preprocessing.imbalance import ImbalanceHandler
from src.preprocessing.feature_selection import FeatureSelector
from src.preprocessing.dimensionality import DimensionalityReducer

print("Libraries successfully imported!")

### 1. Ingesting & Integrating Multimodal Security Telemetry
We load authentication, endpoint, user behavior, and vulnerability data.

In [ ]:
# Generate controlled synthetic security datasets
generator = SyntheticDataGenerator()
datasets = generator.generate_all()

for name, df in datasets.items():
    print(f"Domain: {name:<18} | Records: {len(df):<6} | Columns: {len(df.columns)}")

# Combine datasets into a unified representation
frames = []
for name, df in datasets.items():
    d = df.copy()
    d["source_type"] = name
    frames.append(d)

raw_data = pd.concat(frames, ignore_index=True)
print(f"\nTotal Integrated Records: {len(raw_data):,}")

### 2. Data Cleaning & Feature Transformation

In [ ]:
cleaner = DataCleaner()
clean_data = cleaner.clean(raw_data)
print(f"Records after cleaning and deduplication: {len(clean_data):,}")

transformer = DataTransformer()
transformed_data = transformer.transform(clean_data)
print("Numeric scaling and categorical one-hot encoding complete!")

### 3. Class Imbalance Handling with SMOTE
Cybersecurity telemetry inherently exhibits extreme class imbalance (attack rate typically < 5%). We balance minority attack instances.

In [ ]:
feature_cols = [c for c in transformed_data.columns if c not in [
    'is_attack', 'is_outlier', 'risk_label', 'source_type', 'attack_type', 'timestamp', 'user_id', 'host_id', 'source_ip', 'destination_host', 'process_name', 'parent_process', 'location', 'auth_method'
] and transformed_data[c].dtype in [np.float64, np.float32, np.int64, np.int32]]

X = transformed_data[feature_cols].fillna(0).values
y = transformed_data['is_attack'].astype(int).values

print(f"Original Class Distribution: Benign = {np.sum(y == 0)}, Attack = {np.sum(y == 1)}")

imbalance_handler = ImbalanceHandler()
X_resampled, y_resampled = imbalance_handler.resample(X, y)
print(f"After Resampling: Benign = {np.sum(y_resampled == 0)}, Attack = {np.sum(y_resampled == 1)}")

### 4. Feature Selection (Mutual Information & ANOVA)

In [ ]:
df_X = pd.DataFrame(X_resampled, columns=feature_cols)
df_y = pd.Series(y_resampled)

selector = FeatureSelector()
X_selected = selector.select(df_X, df_y)
scores = selector.get_feature_scores(df_X, df_y).sort_values(ascending=False)

print(f"Selected {len(selector.selected_features)} most informative features.")
scores.head(10).plot(kind='barh', figsize=(10, 5), color='steelblue', title='Top 10 Security Features by Mutual Information')
plt.gca().invert_yaxis()
plt.xlabel('Information Gain')
plt.show()

### 5. Dimensionality Reduction (PCA, LDA, SVD)

In [ ]:
reducer = DimensionalityReducer()
X_pca = reducer.fit_transform(X_selected, df_y)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca['PC1'], X_pca['PC2'], c=y_resampled, cmap='coolwarm', alpha=0.6, s=15)
plt.colorbar(scatter, label='Class (0=Normal, 1=Attack)')
plt.title('2D PCA Projection of Security Feature Space')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True, alpha=0.3)
plt.show()